In [ ]:
# Repository-relative paths for the anonymized reproduction package.
import os
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analysis_code').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository tree.')

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
MODULE_DIR = REPO_ROOT / 'analysis_code' / '05_be_meta_regression'
EXTERNAL_DATA_ROOT = Path(os.environ.get('HEATPA_DATA_ROOT', REPO_ROOT / 'external_data'))


In [ ]:
# -*- coding: utf-8 -*-
"""
Fig. 5d
National exposure-lag response surface

Natural-breaks 180-class version with continuous colorbar:
- Kriging interpolation
- natural breaks color mapping with 180 classes for the heatmap
- continuous gradient colorbar labelled with 5 reference values
- 90% CI-excludes-null points on interpolated grid
- null-effect contour line
- extra y-axis space below p25 and above p95
- extra x-axis space before lag 0 and after max lag
- reference-like landscape layout
- right-side layout: slim colorbar above, legend below, no overlap
- adaptive larger fonts

Input:
    pooled_lag_response_p25.csv
    pooled_lag_response_p50.csv
    pooled_lag_response_p75.csv
    pooled_lag_response_p90.csv
    pooled_lag_response_p95.csv

Output:
    figure5d_lag_response_surface.png
    figure5d_lag_response_surface.svg
    Fig5d_color_boundaries_natural_breaks_180classes.csv
"""

from pathlib import Path
import time
from statistics import NormalDist

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize, BoundaryNorm


# =========================================================
# 0. 计时
# =========================================================

t0 = time.perf_counter()


# =========================================================
# 1. 路径与输出
# =========================================================

# 输入数据路径：使用上一阶段已复制整理到 Fig 5d / Source Data 下的数据。
# 该文件夹应包含：
#   pooled_lag_response_p25.csv
#   pooled_lag_response_p50.csv
#   pooled_lag_response_p75.csv
#   pooled_lag_response_p90.csv
#   pooled_lag_response_p95.csv
DATA_DIR = MODULE_DIR / "data" / "figure5_ab"

# 图件输出路径：保存到 Fig 5d / Figure 文件夹。
FIGURE_OUTPUT_ROOT = MODULE_DIR / "output"
OUT_DIR = FIGURE_OUTPUT_ROOT
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PNG = OUT_DIR / "figure5d_lag_response_surface.png"
OUT_SVG = OUT_DIR / "figure5d_lag_response_surface.svg"

PERCENTILES = [25, 50, 75, 90, 95]


# =========================================================
# 1.1 插值方法与分辨率
# =========================================================

INTERPOLATION_METHOD = "kriging"      # "kriging" or "bilinear"

GRID_MODE = "count"                   # "count" or "step"

# 插值表面分辨率
N_LAG_GRID = 150
N_PERCENTILE_GRID = 150

# 若希望更粗颗粒，可改为：
# N_LAG_GRID = 120
# N_PERCENTILE_GRID = 120

# 若希望更细腻，可改为：
# N_LAG_GRID = 240
# N_PERCENTILE_GRID = 240

# step 模式：插值表面步长
LAG_GRID_STEP = 0.05
PERCENTILE_GRID_STEP = 0.25

# Kriging 参数
KRIGING_VARIANCE_MODEL = "spherical"  # "linear", "power", "gaussian", "spherical", "exponential"
KRIGING_NLAGS = 6
KRIGING_ENABLE_PLOT = False
KRIGING_VERBOSE = False


# =========================================================
# 1.2 x / y 轴留白设置
# =========================================================

# x 轴：在 lag 0 前和最大 lag 后增加绘图范围
X_AXIS_LEFT_PAD = 0.50
X_AXIS_RIGHT_PAD = 0.50

# y 轴：p25 下方和 p95 上方留白
Y_AXIS_LOWER_PAD = 5.0
Y_AXIS_UPPER_PAD = 5.0

Y_AXIS_MIN_LIMIT = 0.0
Y_AXIS_MAX_LIMIT = 100.0


# =========================================================
# 1.3 黑点：90% CI 不跨 null
# =========================================================

SIGNIF_POINT_MODE = "interpolated_ci"    # "original_ci" or "interpolated_ci"

CI_LEVEL_FOR_DOTS = 0.90

# 如果输入文件只有 rr_low / rr_high，而没有 log_rr_se，
# 这里假定 rr_low / rr_high 是 95% CI，用于反推 log_rr_se。
INPUT_CI_LEVEL_FOR_RR_LOW_HIGH = 0.95

# 规则点阵间距
SIGNIF_GRID_LAG_STEP = 1.0
SIGNIF_GRID_PERCENTILE_STEP = 5.0

# 是否显示所有插值候选点
SHOW_INTERPOLATED_CANDIDATE_DOTS = False
INTERPOLATED_CANDIDATE_DOT_SIZE = 5
INTERPOLATED_CANDIDATE_DOT_ALPHA = 0.25

# 是否显示原始估计点
SHOW_ALL_ORIGINAL_GRID_POINTS = False
ALL_ORIGINAL_DOT_SIZE = 9
ALL_ORIGINAL_DOT_ALPHA = 0.35

# 黑点大小
ORIGINAL_SIGNIF_DOT_SIZE = 16
INTERPOLATED_SIGNIF_DOT_SIZE = 8.5

ORIGINAL_SIGNIF_LABEL = f"{int(CI_LEVEL_FOR_DOTS * 100)}% CI excludes null"
INTERPOLATED_SIGNIF_LABEL = f"{int(CI_LEVEL_FOR_DOTS * 100)}% CI excludes null"


# =========================================================
# 1.4 颜色设置：60 层自然断裂 + 连续色带 legend/colorbar
# =========================================================

COLOR_MAPPING_MODE = "natural_breaks"

# 180 表示热力图内部使用 180 个自然断裂颜色等级。
# 视觉上接近连续，但能增强局部色彩差异。
N_COLOR_CLASSES = 180

# 颜色范围。colorbar 固定标注 -10, -5, 0, 5, 10。
COLOR_LIMIT = 10.0
COLOR_VMIN = -COLOR_LIMIT
COLOR_VMAX = COLOR_LIMIT

# 是否将热力图值限制在 COLOR_VMIN ~ COLOR_VMAX 内。
CLIP_COLOR_VALUES = True

# 自然断裂取样来源：
# "interpolated_sampled"：推荐，速度较快且代表热力图
# "original"：最快，但只基于原始点
# "interpolated_full"：最慢，不推荐用于 180 层 Jenks
COLOR_BREAK_SOURCE = "interpolated_sampled"

# 自然断裂抽样数量。180 层 Jenks 不宜太大，否则纯 Python 会较慢。
# 若运行慢，改为 500；若想更稳定，改为 1000–1500。
COLOR_BREAK_MAX_SAMPLE = 800

# 强制 0 作为中间断裂点：负值 90 层、正值 90 层。
FORCE_ZERO_BREAK = True

# 右侧 colorbar 是否显示为连续渐变。True 时 colorbar 与参考图一致。
COLORBAR_AS_CONTINUOUS_GRADIENT = True

# colorbar 只标注 5 个数字。
COLORBAR_TICK_VALUES = [-10, -5, 0, 5, 10]
COLORBAR_N_TICKS = 5
COLORBAR_EXTEND = "neither"

# 使用参考图风格的蓝-白-红色带。
USE_REFERENCE_CMAP = True
CMAP = "RdBu_r"

# 保存自然断裂边界表。
SAVE_COLOR_BOUNDARIES_CSV = True

# =========================================================
# 1.4.1 白色虚线等高线 + 透明度
# =========================================================

# 热力图整体透明度（0~1）
HEATMAP_ALPHA = 0.86

# 右侧连续色带 / colorbar 的透明度（0~1）
COLORBAR_ALPHA = 0.86

# 是否显示白色虚线等高线
SHOW_WHITE_CONTOURS = True

# 等高线层级控制方式：
# "count"：按数量平均生成等高线
# "step"：按固定步长生成等高线
WHITE_CONTOUR_LEVEL_MODE = "count"

# 当 WHITE_CONTOUR_LEVEL_MODE = "count" 时生效
WHITE_CONTOUR_N_LEVELS = 10

# 当 WHITE_CONTOUR_LEVEL_MODE = "step" 时生效
WHITE_CONTOUR_STEP = 2.0

# 是否排除 0 等值线，避免与黑色 null-effect 虚线重叠
WHITE_CONTOUR_EXCLUDE_ZERO = True

WHITE_CONTOUR_COLOR = "white"
WHITE_CONTOUR_LINEWIDTH = 1.05
WHITE_CONTOUR_LINESTYLE = "--"
WHITE_CONTOUR_ALPHA = 0.72


# =========================================================
# 1.5 0% null effect 虚线
# =========================================================

NULL_LINE_MODE = "contour"

NULL_CONTOUR_COLOR = "black"
NULL_CONTOUR_LINEWIDTH = 1.5
NULL_CONTOUR_LINESTYLE = "--"
NULL_CONTOUR_ALPHA = 0.90


# =========================================================
# 1.6 输出与加速选项
# =========================================================

SAVE_LARGE_SURFACE_CSV = False
SAVE_SOURCE_SUMMARY_CSV = True
SAVE_INTERPOLATED_CI_POINTS_CSV = True


# =========================================================
# 1.7 图像样式与参考图布局
# =========================================================

# 参考图接近中等横向比例，不建议过宽
FIGSIZE = (9.4, 5.9)
DPI = 600

FONT_FAMILY = "Arial"

# 字体自适应放大
AUTO_FONT_SCALE = True
REFERENCE_FIG_WIDTH = 8.8
MIN_FONT_SCALE = 1.06
MAX_FONT_SCALE = 1.28

# 基础字体大小
TITLE_SIZE_BASE = 14.0
SUBTITLE_SIZE_BASE = 11.0
AXIS_LABEL_SIZE_BASE = 11.5
TICK_SIZE_BASE = 10.0
CBAR_LABEL_SIZE_BASE = 9.5
PANEL_LABEL_SIZE_BASE = 20.0
LEGEND_FONT_SIZE_BASE = 8.2

SHOW_PANEL_LABEL = True
PANEL_LABEL = "b"

# 主图框比例：height / width
SET_MAIN_AX_BOX_ASPECT = True
MAIN_AX_BOX_ASPECT = 0.72

# 右侧整体面板宽度
RIGHT_PANEL_WIDTH_RATIO = 0.22
RIGHT_PANEL_WSPACE = 0.075

# 整体边距
FIG_LEFT = 0.085
FIG_RIGHT = 0.965
FIG_BOTTOM = 0.135
FIG_TOP = 0.865

# 右侧 colorbar 和 legend 的位置，均为 right_ax 内部的相对坐标
# [x0, y0, width, height]
CBAR_BOX = [0.18, 0.42, 0.16, 0.47]
LEGEND_BOX = [0.00, 0.12, 0.98, 0.22]

# legend 样式
LEGEND_HANDLE_LENGTH = 2.1
LEGEND_LABEL_SPACING = 0.75
LEGEND_HANDLE_TEXT_PAD = 0.65

NULL_EFFECT_LEGEND_LABEL = "0% null effect"
# 如果希望更完整，可以改成：
# NULL_EFFECT_LEGEND_LABEL = "0% null effect\n(RR = 1 / log-RR = 0)"


# =========================================================
# 1.8 字体自适应计算
# =========================================================

def _font_scale():
    if not AUTO_FONT_SCALE:
        return 1.0
    scale = FIGSIZE[0] / REFERENCE_FIG_WIDTH
    scale = max(MIN_FONT_SCALE, min(MAX_FONT_SCALE, scale))
    return scale


_FONT_SCALE = _font_scale()

TITLE_SIZE = TITLE_SIZE_BASE * _FONT_SCALE
SUBTITLE_SIZE = SUBTITLE_SIZE_BASE * _FONT_SCALE
AXIS_LABEL_SIZE = AXIS_LABEL_SIZE_BASE * _FONT_SCALE
TICK_SIZE = TICK_SIZE_BASE * _FONT_SCALE
CBAR_LABEL_SIZE = CBAR_LABEL_SIZE_BASE * _FONT_SCALE
PANEL_LABEL_SIZE = PANEL_LABEL_SIZE_BASE * _FONT_SCALE
LEGEND_FONT_SIZE = LEGEND_FONT_SIZE_BASE * _FONT_SCALE


# =========================================================
# 2. 工具函数
# =========================================================

def z_value_for_ci(ci_level: float) -> float:
    """
    双侧置信区间对应的 z 值。
    90% CI -> 1.64485
    95% CI -> 1.95996
    """
    return NormalDist().inv_cdf(0.5 + ci_level / 2.0)


def find_percentile_file(data_dir: Path, p: int) -> Path:
    exact = data_dir / f"pooled_lag_response_p{p}.csv"
    if exact.exists():
        return exact

    candidates = list(data_dir.glob(f"pooled_lag_response_p{p}*.csv"))
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"未找到 p{p} 文件：{exact.name} 或 pooled_lag_response_p{p}*.csv"
        )

    return candidates[0]


def load_lag_profiles(data_dir: Path, percentiles) -> pd.DataFrame:
    """
    读取 p25/p50/p75/p90/p95 的 pooled lag-response 文件。

    黑点使用的 CI 通过 CI_LEVEL_FOR_DOTS 控制。
    若文件有 log_rr_se，直接用 log_rr_se。
    若没有 log_rr_se，但有 rr_low / rr_high，则假定它们是 INPUT_CI_LEVEL_FOR_RR_LOW_HIGH
    对应的 CI，并反推 log_rr_se。
    """
    dfs = []

    z_input = z_value_for_ci(INPUT_CI_LEVEL_FOR_RR_LOW_HIGH)
    z_target = z_value_for_ci(CI_LEVEL_FOR_DOTS)

    for p in percentiles:
        f = find_percentile_file(data_dir, p)
        df = pd.read_csv(f)

        required_cols = ["lag", "rr"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"{f.name} 缺少必要列：{missing}")

        df = df.copy()
        df["percentile"] = p
        df["source_file"] = f.name

        df["lag"] = pd.to_numeric(df["lag"], errors="coerce")
        df["rr"] = pd.to_numeric(df["rr"], errors="coerce")

        if "log_rr" in df.columns:
            df["log_rr"] = pd.to_numeric(df["log_rr"], errors="coerce")
        else:
            df["log_rr"] = np.log(df["rr"])

        if "log_rr_se" in df.columns:
            df["log_rr_se"] = pd.to_numeric(df["log_rr_se"], errors="coerce")
            df["ci_source"] = "provided_log_rr_se"

        elif ("rr_low" in df.columns) and ("rr_high" in df.columns):
            df["rr_low"] = pd.to_numeric(df["rr_low"], errors="coerce")
            df["rr_high"] = pd.to_numeric(df["rr_high"], errors="coerce")

            valid_ci = (
                (df["rr_low"] > 0) &
                (df["rr_high"] > 0) &
                np.isfinite(df["rr_low"]) &
                np.isfinite(df["rr_high"])
            )

            df["log_rr_se"] = np.nan
            df.loc[valid_ci, "log_rr_se"] = (
                np.log(df.loc[valid_ci, "rr_high"]) -
                np.log(df.loc[valid_ci, "rr_low"])
            ) / (2.0 * z_input)

            df["ci_source"] = (
                f"reconstructed_log_rr_se_from_"
                f"{int(INPUT_CI_LEVEL_FOR_RR_LOW_HIGH * 100)}CI_rr_low_rr_high"
            )

        else:
            df["log_rr_se"] = np.nan
            df["ci_source"] = "not_available"

        df["log_rr_low_for_dots"] = df["log_rr"] - z_target * df["log_rr_se"]
        df["log_rr_high_for_dots"] = df["log_rr"] + z_target * df["log_rr_se"]

        df["rr_low_for_dots"] = np.exp(df["log_rr_low_for_dots"])
        df["rr_high_for_dots"] = np.exp(df["log_rr_high_for_dots"])

        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    out["percentile"] = pd.to_numeric(out["percentile"], errors="coerce")
    out["log_rr_se"] = pd.to_numeric(out["log_rr_se"], errors="coerce")

    out = out.dropna(subset=["lag", "percentile", "rr", "log_rr"])

    out["pa_change_pct"] = (out["rr"] - 1.0) * 100.0

    out["ci_excludes_null"] = (
        (
            (out["rr_low_for_dots"] > 1.0) |
            (out["rr_high_for_dots"] < 1.0)
        )
        & np.isfinite(out["rr_low_for_dots"])
        & np.isfinite(out["rr_high_for_dots"])
    )

    return out


def make_matrix(df: pd.DataFrame, value_col: str):
    mat = (
        df.pivot_table(
            index="percentile",
            columns="lag",
            values=value_col,
            aggfunc="mean"
        )
        .sort_index()
        .sort_index(axis=1)
    )

    x = mat.columns.to_numpy(dtype=float)
    y = mat.index.to_numpy(dtype=float)
    z = mat.to_numpy(dtype=float)

    return x, y, z


def build_interpolation_grid(x_lag, y_pct):
    x_min_raw = float(np.nanmin(x_lag))
    x_max_raw = float(np.nanmax(x_lag))

    x_min = x_min_raw - X_AXIS_LEFT_PAD
    x_max = x_max_raw + X_AXIS_RIGHT_PAD

    y_min_raw = float(np.nanmin(y_pct))
    y_max_raw = float(np.nanmax(y_pct))

    y_min = max(Y_AXIS_MIN_LIMIT, y_min_raw - Y_AXIS_LOWER_PAD)
    y_max = min(Y_AXIS_MAX_LIMIT, y_max_raw + Y_AXIS_UPPER_PAD)

    if GRID_MODE == "count":
        x_plot = np.linspace(x_min, x_max, N_LAG_GRID)
        y_plot = np.linspace(y_min, y_max, N_PERCENTILE_GRID)

    elif GRID_MODE == "step":
        x_plot = np.arange(x_min, x_max + LAG_GRID_STEP * 0.5, LAG_GRID_STEP)
        y_plot = np.arange(y_min, y_max + PERCENTILE_GRID_STEP * 0.5, PERCENTILE_GRID_STEP)

        x_plot = x_plot[x_plot <= x_max + 1e-9]
        y_plot = y_plot[y_plot <= y_max + 1e-9]

    else:
        raise ValueError("GRID_MODE 只能是 'count' 或 'step'")

    return x_plot, y_plot


def bilinear_interpolate_grid(x_old, y_old, z_old, x_new, y_new):
    z_x = np.vstack([
        np.interp(x_new, x_old, row)
        for row in z_old
    ])

    z_new = np.vstack([
        np.interp(y_new, y_old, z_x[:, j])
        for j in range(z_x.shape[1])
    ]).T

    return z_new


def normalize_to_unit(values, vmin=None, vmax=None):
    values = np.asarray(values, dtype=float)

    if vmin is None:
        vmin = np.nanmin(values)
    if vmax is None:
        vmax = np.nanmax(values)

    if np.isclose(vmax, vmin):
        return np.zeros_like(values), vmin, vmax

    return (values - vmin) / (vmax - vmin), vmin, vmax


def kriging_interpolate_grid(df, value_col, x_new, y_new):
    """
    Ordinary Kriging 二维插值。

    lag 和 percentile 数值尺度不同，因此先标准化到 0–1。
    注意：如果 x_new / y_new 超出原始范围，该区域属于外推绘图范围。
    """
    try:
        from pykrige.ok import OrdinaryKriging
    except ImportError as e:
        raise ImportError(
            "当前环境未安装 PyKrige。请先运行：pip install pykrige"
        ) from e

    work = df[["lag", "percentile", value_col]].copy()
    work = work.dropna()

    if len(work) < 5:
        raise ValueError(f"{value_col} 可用于 Kriging 的有效点太少：{len(work)}")

    work = (
        work.groupby(["lag", "percentile"], as_index=False)[value_col]
        .mean()
    )

    x = work["lag"].to_numpy(dtype=float)
    y = work["percentile"].to_numpy(dtype=float)
    z = work[value_col].to_numpy(dtype=float)

    x_norm, x_min, x_max = normalize_to_unit(x)
    y_norm, y_min, y_max = normalize_to_unit(y)

    x_new_norm, _, _ = normalize_to_unit(x_new, x_min, x_max)
    y_new_norm, _, _ = normalize_to_unit(y_new, y_min, y_max)

    OK = OrdinaryKriging(
        x_norm,
        y_norm,
        z,
        variogram_model=KRIGING_VARIANCE_MODEL,
        nlags=KRIGING_NLAGS,
        verbose=KRIGING_VERBOSE,
        enable_plotting=KRIGING_ENABLE_PLOT
    )

    z_pred, z_var = OK.execute("grid", x_new_norm, y_new_norm)

    z_pred = np.ma.filled(z_pred, np.nan)
    z_var = np.ma.filled(z_var, np.nan)

    return np.asarray(z_pred, dtype=float), np.asarray(z_var, dtype=float)


def build_surface_for_value(df, value_col, x_lag, y_pct, x_plot, y_plot, method):
    if method == "kriging":
        z_surface, z_var = kriging_interpolate_grid(
            df=df,
            value_col=value_col,
            x_new=x_plot,
            y_new=y_plot
        )
        return z_surface, z_var

    elif method == "bilinear":
        _, _, z_mat = make_matrix(df, value_col)
        z_surface = bilinear_interpolate_grid(
            x_lag,
            y_pct,
            z_mat,
            x_plot,
            y_plot
        )
        return z_surface, None

    else:
        raise ValueError("method 只能是 'kriging' 或 'bilinear'")


def draw_null_effect_contour(
    ax,
    X,
    Y,
    z_plot,
    color="black",
    linewidth=1.2,
    linestyle="--",
    alpha=1.0
):
    """
    z_plot = Change in physical activity (%)
    z_plot = 0 等价于 RR = 1，也等价于 log-RR = 0。
    """
    if np.nanmin(z_plot) <= 0 <= np.nanmax(z_plot):
        ax.contour(
            X,
            Y,
            z_plot,
            levels=[0],
            colors=color,
            linewidths=linewidth,
            linestyles=linestyle,
            alpha=alpha,
            zorder=6
        )


def build_regular_dot_grid(x_min, x_max, y_min, y_max, lag_step=1.0, percentile_step=5.0):
    """
    黑点候选网格限制在原始 lag 和 p25–p95 范围内，
    不进入 x/y 轴留白区域。
    """
    x_start = np.ceil(x_min / lag_step) * lag_step
    x_end = np.floor(x_max / lag_step) * lag_step

    y_start = np.ceil(y_min / percentile_step) * percentile_step
    y_end = np.floor(y_max / percentile_step) * percentile_step

    xs = np.arange(x_start, x_end + lag_step * 0.5, lag_step)
    ys = np.arange(y_start, y_end + percentile_step * 0.5, percentile_step)

    xs = xs[(xs >= x_min - 1e-9) & (xs <= x_max + 1e-9)]
    ys = ys[(ys >= y_min - 1e-9) & (ys <= y_max + 1e-9)]

    XX, YY = np.meshgrid(xs, ys)

    return XX.ravel(), YY.ravel()


def sample_surface_at_points_fast(x_grid, y_grid, z_grid, x_points, y_points):
    """
    在规则插值面上向量化采样。
    """
    x_grid = np.asarray(x_grid, dtype=float)
    y_grid = np.asarray(y_grid, dtype=float)
    z_grid = np.asarray(z_grid, dtype=float)

    x_points = np.asarray(x_points, dtype=float)
    y_points = np.asarray(y_points, dtype=float)

    values = np.full(len(x_points), np.nan, dtype=float)

    valid = (
        (x_points >= x_grid.min()) &
        (x_points <= x_grid.max()) &
        (y_points >= y_grid.min()) &
        (y_points <= y_grid.max())
    )

    if not np.any(valid):
        return values

    xp = x_points[valid]
    yp = y_points[valid]

    ix = np.searchsorted(x_grid, xp, side="right") - 1
    iy = np.searchsorted(y_grid, yp, side="right") - 1

    ix = np.clip(ix, 0, len(x_grid) - 2)
    iy = np.clip(iy, 0, len(y_grid) - 2)

    x0 = x_grid[ix]
    x1 = x_grid[ix + 1]
    y0 = y_grid[iy]
    y1 = y_grid[iy + 1]

    tx = np.where(np.isclose(x1, x0), 0.0, (xp - x0) / (x1 - x0))
    ty = np.where(np.isclose(y1, y0), 0.0, (yp - y0) / (y1 - y0))

    z00 = z_grid[iy, ix]
    z10 = z_grid[iy, ix + 1]
    z01 = z_grid[iy + 1, ix]
    z11 = z_grid[iy + 1, ix + 1]

    sampled = (
        (1 - tx) * (1 - ty) * z00 +
        tx * (1 - ty) * z10 +
        (1 - tx) * ty * z01 +
        tx * ty * z11
    )

    values[valid] = sampled

    return values


def save_surface_csv(z, y_plot, x_plot, out_csv):
    out = pd.DataFrame(z, index=y_plot, columns=x_plot)
    out.index.name = "percentile"
    out.to_csv(out_csv, encoding="utf-8-sig")


def get_colormap():
    """
    参考图风格蓝-白-红连续色带。
    """
    if USE_REFERENCE_CMAP:
        return LinearSegmentedColormap.from_list(
            "reference_blue_white_red",
            [
                "#08306B",
                "#08519C",
                "#2171B5",
                "#6BAED6",
                "#D6EAF8",
                "#F7F7F7",
                "#FADBD8",
                "#F1948A",
                "#E74C3C",
                "#B03A2E",
                "#7B0000",
            ],
            N=256
        )
    else:
        return plt.get_cmap(CMAP)


def make_alpha_colormap(base_cmap, alpha=1.0, name_suffix=""):
    """
    给 colormap 统一附加透明度，分别用于热力图与右侧 colorbar。
    """
    alpha = float(np.clip(alpha, 0.0, 1.0))
    colors = base_cmap(np.linspace(0, 1, 256))
    colors[:, 3] = alpha
    return LinearSegmentedColormap.from_list(
        f"{getattr(base_cmap, 'name', 'cmap')}_{name_suffix}_a{int(alpha * 1000)}",
        colors,
        N=256
    )


def build_white_contour_levels(z, mode="count", n_levels=10, step=2.0, exclude_zero=True):
    """
    根据当前热力图显示值构建白色虚线等高线层级。
    """
    z = np.asarray(z, dtype=float)
    z = z[np.isfinite(z)]

    if z.size == 0:
        return np.array([])

    zmin = float(np.nanmin(z))
    zmax = float(np.nanmax(z))

    if np.isclose(zmin, zmax):
        return np.array([])

    if mode == "count":
        n_levels = int(max(1, n_levels))
        levels = np.linspace(zmin, zmax, n_levels + 2)[1:-1]
    elif mode == "step":
        step = float(step)
        if step <= 0:
            raise ValueError("WHITE_CONTOUR_STEP 必须大于 0")
        start = np.ceil(zmin / step) * step
        end = np.floor(zmax / step) * step
        levels = np.arange(start, end + step * 0.5, step)
    else:
        raise ValueError('WHITE_CONTOUR_LEVEL_MODE 只能是 "count" 或 "step"')

    levels = np.asarray(levels, dtype=float)
    levels = levels[np.isfinite(levels)]

    if exclude_zero and levels.size > 0:
        tol = max(1e-10, (max(abs(zmin), abs(zmax)) + 1.0) * 1e-10)
        levels = levels[np.abs(levels) > tol]

    return levels


def format_color_label(v):
    """
    colorbar 标签格式。
    """
    if abs(v) < 0.005:
        return "0"
    elif abs(v) < 1:
        return f"{v:.2f}"
    else:
        return f"{v:.0f}" if abs(v) >= 10 else f"{v:.1f}"


# =========================================================
# 2.1 自然断裂法工具函数
# =========================================================

def get_break_values(z_for_color, z_change_for_color, source="interpolated_sampled", max_sample=800):
    """
    获取用于颜色分级的数值。
    """
    if source == "original":
        values = np.asarray(z_change_for_color, dtype=float).ravel()

    elif source == "interpolated_full":
        values = np.asarray(z_for_color, dtype=float).ravel()

    elif source == "interpolated_sampled":
        values = np.asarray(z_for_color, dtype=float).ravel()
        values = values[np.isfinite(values)]

        if len(values) > max_sample:
            # 均匀抽样，保证每次结果可复现
            idx = np.linspace(0, len(values) - 1, max_sample).astype(int)
            values = values[idx]

    else:
        raise ValueError("COLOR_BREAK_SOURCE 只能是 'original', 'interpolated_full', 或 'interpolated_sampled'")

    values = values[np.isfinite(values)]
    return values


def jenks_breaks_1d(values, n_classes):
    """
    Jenks natural breaks 自然断裂法。
    使用动态规划。建议输入抽样后的数据，避免运行过慢。
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = np.sort(values)

    if len(values) == 0:
        raise ValueError("自然断裂法没有可用数值。")

    unique_values = np.unique(values)

    if len(unique_values) <= 2:
        return np.linspace(values.min(), values.max(), n_classes + 1)

    n_classes = int(min(n_classes, len(unique_values)))
    n_data = len(values)

    lower = np.zeros((n_data + 1, n_classes + 1), dtype=int)
    var = np.full((n_data + 1, n_classes + 1), np.inf)

    for i in range(1, n_classes + 1):
        lower[1, i] = 1
        var[1, i] = 0.0

    for l in range(2, n_data + 1):
        s1 = 0.0
        s2 = 0.0
        w = 0.0

        for m in range(1, l + 1):
            i3 = l - m + 1
            val = values[i3 - 1]

            s1 += val
            s2 += val * val
            w += 1.0

            variance = s2 - (s1 * s1) / w
            i4 = i3 - 1

            if i4 != 0:
                for j in range(2, n_classes + 1):
                    test = variance + var[i4, j - 1]
                    if var[l, j] >= test:
                        lower[l, j] = i3
                        var[l, j] = test

        lower[l, 1] = 1
        var[l, 1] = variance

    breaks = np.zeros(n_classes + 1)
    breaks[n_classes] = values[-1]
    k = n_data

    for c in range(n_classes, 1, -1):
        idx = int(lower[k, c] - 2)
        breaks[c - 1] = values[idx]
        k = int(lower[k, c] - 1)

    breaks[0] = values[0]

    return breaks


def make_strictly_increasing_breaks(breaks, vmin, vmax):
    """
    BoundaryNorm 要求断点严格递增。
    如果自然断裂产生重复断点，则做极小调整；
    若重复过多，则退回线性分级。
    """
    breaks = np.asarray(breaks, dtype=float)
    breaks = np.sort(breaks)

    if len(np.unique(breaks)) < max(3, len(breaks) * 0.6):
        return np.linspace(vmin, vmax, len(breaks))

    breaks[0] = vmin
    breaks[-1] = vmax

    eps = max(1e-8, (vmax - vmin) * 1e-8)

    for i in range(1, len(breaks)):
        if breaks[i] <= breaks[i - 1]:
            breaks[i] = breaks[i - 1] + eps

    if breaks[-1] > vmax:
        return np.linspace(vmin, vmax, len(breaks))

    breaks[0] = vmin
    breaks[-1] = vmax

    return breaks


def make_diverging_natural_breaks(values, n_classes=60, vmin_fixed=-10, vmax_fixed=10, force_zero=True):
    """
    发散型自然断裂法分级。

    n_classes=180 时：负值 30 类、正值 30 类，并强制 0 作为中间断裂点。
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError("自然断裂法没有可用数值。")

    n_classes = int(n_classes)

    if n_classes < 2:
        raise ValueError("N_COLOR_CLASSES 至少应为 2。")

    vmin = float(vmin_fixed)
    vmax = float(vmax_fixed)

    if not (vmin < 0 < vmax):
        raise ValueError("自然断裂法当前版本要求 vmin < 0 < vmax。")

    n_neg_classes = n_classes // 2
    n_pos_classes = n_classes - n_neg_classes

    neg = values[(values < 0) & np.isfinite(values)]
    pos = values[(values > 0) & np.isfinite(values)]

    if len(neg) >= max(3, n_neg_classes):
        neg_breaks = jenks_breaks_1d(neg, n_neg_classes)
        neg_breaks[0] = vmin
        neg_breaks[-1] = 0.0 if force_zero else neg_breaks[-1]
    else:
        neg_breaks = np.linspace(vmin, 0.0, n_neg_classes + 1)

    if len(pos) >= max(3, n_pos_classes):
        pos_breaks = jenks_breaks_1d(pos, n_pos_classes)
        pos_breaks[0] = 0.0 if force_zero else pos_breaks[0]
        pos_breaks[-1] = vmax
    else:
        pos_breaks = np.linspace(0.0, vmax, n_pos_classes + 1)

    breaks = np.concatenate([neg_breaks, pos_breaks[1:]])
    breaks = make_strictly_increasing_breaks(breaks, vmin=vmin, vmax=vmax)

    return breaks


# =========================================================
# 3. 读取数据
# =========================================================

print("Step 1/7: loading data...")

df = load_lag_profiles(DATA_DIR, PERCENTILES)

source_csv = OUT_DIR / "Fig5d_source_data_supplied_percentile_lag_profiles.csv"

if SAVE_SOURCE_SUMMARY_CSV:
    df.to_csv(source_csv, index=False, encoding="utf-8-sig")

x_lag, y_pct, z_change = make_matrix(df, "pa_change_pct")

sig_df = df[df["ci_excludes_null"]].copy()

x_plot, y_plot = build_interpolation_grid(x_lag, y_pct)

print("Interpolation grid:")
print(f"  Lag points: {len(x_plot)}")
print(f"  Percentile points: {len(y_plot)}")
print(f"  Total cells: {len(x_plot) * len(y_plot)}")
print(f"  X-axis range: {x_plot.min():.2f} to {x_plot.max():.2f}")
print(f"  Y-axis range: {y_plot.min():.2f} to {y_plot.max():.2f}")


# =========================================================
# 4. 构建主效应插值面
# =========================================================

print("Step 2/7: interpolating main response surface...")

z_plot, z_var = build_surface_for_value(
    df=df,
    value_col="pa_change_pct",
    x_lag=x_lag,
    y_pct=y_pct,
    x_plot=x_plot,
    y_plot=y_plot,
    method=INTERPOLATION_METHOD
)

surface_csv = OUT_DIR / "Fig5d_interpolated_surface_for_plotting.csv"

if SAVE_LARGE_SURFACE_CSV:
    save_surface_csv(z_plot, y_plot, x_plot, surface_csv)

if INTERPOLATION_METHOD == "kriging" and z_var is not None and SAVE_LARGE_SURFACE_CSV:
    kriging_var_csv = OUT_DIR / "Fig5d_kriging_variance_surface_pa_change_pct.csv"
    save_surface_csv(z_var, y_plot, x_plot, kriging_var_csv)
else:
    kriging_var_csv = None


# =========================================================
# 5. 插值目标 CI 面，并生成规则黑点
# =========================================================

print(f"Step 3/7: interpolating {int(CI_LEVEL_FOR_DOTS * 100)}% CI surfaces and generating dot grid...")

interp_candidate_df = pd.DataFrame()
interp_sig_df = pd.DataFrame()

if SIGNIF_POINT_MODE == "interpolated_ci":

    if df["log_rr_low_for_dots"].notna().sum() < 5 or df["log_rr_high_for_dots"].notna().sum() < 5:
        raise ValueError(
            f"当前数据无法进行 {int(CI_LEVEL_FOR_DOTS * 100)}% CI 插值判断："
            "缺少足够的 log_rr_se，且无法由 rr_low / rr_high 反推。"
        )

    log_rr_low_plot, log_rr_low_var = build_surface_for_value(
        df=df,
        value_col="log_rr_low_for_dots",
        x_lag=x_lag,
        y_pct=y_pct,
        x_plot=x_plot,
        y_plot=y_plot,
        method=INTERPOLATION_METHOD
    )

    log_rr_high_plot, log_rr_high_var = build_surface_for_value(
        df=df,
        value_col="log_rr_high_for_dots",
        x_lag=x_lag,
        y_pct=y_pct,
        x_plot=x_plot,
        y_plot=y_plot,
        method=INTERPOLATION_METHOD
    )

    if SAVE_LARGE_SURFACE_CSV:
        log_low_csv = OUT_DIR / f"Fig5d_interpolated_log_rr_low_{int(CI_LEVEL_FOR_DOTS * 100)}CI_surface.csv"
        log_high_csv = OUT_DIR / f"Fig5d_interpolated_log_rr_high_{int(CI_LEVEL_FOR_DOTS * 100)}CI_surface.csv"
        save_surface_csv(log_rr_low_plot, y_plot, x_plot, log_low_csv)
        save_surface_csv(log_rr_high_plot, y_plot, x_plot, log_high_csv)

    dot_x, dot_y = build_regular_dot_grid(
        x_min=x_lag.min(),
        x_max=x_lag.max(),
        y_min=y_pct.min(),
        y_max=y_pct.max(),
        lag_step=SIGNIF_GRID_LAG_STEP,
        percentile_step=SIGNIF_GRID_PERCENTILE_STEP
    )

    dot_log_low = sample_surface_at_points_fast(
        x_grid=x_plot,
        y_grid=y_plot,
        z_grid=log_rr_low_plot,
        x_points=dot_x,
        y_points=dot_y
    )

    dot_log_high = sample_surface_at_points_fast(
        x_grid=x_plot,
        y_grid=y_plot,
        z_grid=log_rr_high_plot,
        x_points=dot_x,
        y_points=dot_y
    )

    dot_sig = (
        ((dot_log_low > 0.0) | (dot_log_high < 0.0))
        & np.isfinite(dot_log_low)
        & np.isfinite(dot_log_high)
    )

    interp_candidate_df = pd.DataFrame({
        "lag": dot_x,
        "percentile": dot_y,
        "log_rr_low_interp": dot_log_low,
        "log_rr_high_interp": dot_log_high,
        "rr_low_interp": np.exp(dot_log_low),
        "rr_high_interp": np.exp(dot_log_high),
        "ci_level": CI_LEVEL_FOR_DOTS,
        "ci_excludes_null_interp": dot_sig
    })

    interp_sig_df = interp_candidate_df[
        interp_candidate_df["ci_excludes_null_interp"]
    ].copy()

    interp_points_csv = OUT_DIR / f"Fig5d_interpolated_{int(CI_LEVEL_FOR_DOTS * 100)}CI_excludes_null_points.csv"

    if SAVE_INTERPOLATED_CI_POINTS_CSV:
        interp_candidate_df.to_csv(interp_points_csv, index=False, encoding="utf-8-sig")

elif SIGNIF_POINT_MODE == "original_ci":
    interp_points_csv = None

else:
    raise ValueError("SIGNIF_POINT_MODE 只能是 'original_ci' 或 'interpolated_ci'")


# =========================================================
# 6. 60 层自然断裂色彩映射 + 连续 colorbar
# =========================================================

print("Step 4/7: setting natural-breaks color mapping with continuous colorbar...")

cmap_base = get_colormap()

if CLIP_COLOR_VALUES:
    z_for_color = np.clip(z_plot, COLOR_VMIN, COLOR_VMAX)
    z_change_for_color = np.clip(z_change, COLOR_VMIN, COLOR_VMAX)
else:
    z_for_color = z_plot.copy()
    z_change_for_color = z_change.copy()

break_values = get_break_values(
    z_for_color=z_for_color,
    z_change_for_color=z_change_for_color,
    source=COLOR_BREAK_SOURCE,
    max_sample=COLOR_BREAK_MAX_SAMPLE
)

color_boundaries = make_diverging_natural_breaks(
    values=break_values,
    n_classes=N_COLOR_CLASSES,
    vmin_fixed=COLOR_VMIN,
    vmax_fixed=COLOR_VMAX,
    force_zero=FORCE_ZERO_BREAK
)

# 热力图内部使用自然断裂映射
n_bins = len(color_boundaries) - 1
cmap_for_mesh = LinearSegmentedColormap.from_list(
    f"{cmap_base.name}_{n_bins}", cmap_base(np.linspace(0.0, 1.0, n_bins)), N=n_bins
)
norm_for_mesh = BoundaryNorm(color_boundaries, cmap_for_mesh.N, clip=True)

# colorbar 使用连续映射，并固定 5 个标注
cmap_for_cbar = cmap_base
norm_for_cbar = Normalize(vmin=COLOR_VMIN, vmax=COLOR_VMAX, clip=True)

# 分别为热力图与 colorbar 添加可调透明度
cmap_for_mesh = make_alpha_colormap(cmap_for_mesh, alpha=HEATMAP_ALPHA, name_suffix="mesh")
cmap_for_cbar = make_alpha_colormap(cmap_for_cbar, alpha=COLORBAR_ALPHA, name_suffix="cbar")

if SAVE_COLOR_BOUNDARIES_CSV:
    color_break_csv = OUT_DIR / f"Fig5d_color_boundaries_natural_breaks_{N_COLOR_CLASSES}classes.csv"
    pd.DataFrame({
        "boundary_index": np.arange(len(color_boundaries)),
        "boundary_value": color_boundaries
    }).to_csv(color_break_csv, index=False, encoding="utf-8-sig")
else:
    color_break_csv = None


# =========================================================
# 7. 绘图：参考图比例 + 独立右侧面板
# =========================================================

print("Step 5/7: plotting figure...")

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "none"

X, Y = np.meshgrid(x_plot, y_plot)

fig = plt.figure(figsize=FIGSIZE)

gs = fig.add_gridspec(
    nrows=1,
    ncols=2,
    width_ratios=[1.0, RIGHT_PANEL_WIDTH_RATIO],
    wspace=RIGHT_PANEL_WSPACE
)

ax = fig.add_subplot(gs[0, 0])

right_ax = fig.add_subplot(gs[0, 1])
right_ax.axis("off")

# 右侧：上方细长连续色带，下方 legend
cax = right_ax.inset_axes(CBAR_BOX)
leg_ax = right_ax.inset_axes(LEGEND_BOX)
leg_ax.axis("off")

if SET_MAIN_AX_BOX_ASPECT:
    ax.set_box_aspect(MAIN_AX_BOX_ASPECT)

mesh = ax.pcolormesh(
    X,
    Y,
    z_for_color,
    cmap=cmap_for_mesh,
    norm=norm_for_mesh,
    shading="auto"
)

# 白色虚线等高线（与热力图内部显色值一致）
if SHOW_WHITE_CONTOURS:
    white_contour_levels = build_white_contour_levels(
        z_for_color,
        mode=WHITE_CONTOUR_LEVEL_MODE,
        n_levels=WHITE_CONTOUR_N_LEVELS,
        step=WHITE_CONTOUR_STEP,
        exclude_zero=WHITE_CONTOUR_EXCLUDE_ZERO
    )

    if len(white_contour_levels) > 0:
        ax.contour(
            X,
            Y,
            z_for_color,
            levels=white_contour_levels,
            colors=WHITE_CONTOUR_COLOR,
            linewidths=WHITE_CONTOUR_LINEWIDTH,
            linestyles=WHITE_CONTOUR_LINESTYLE,
            alpha=WHITE_CONTOUR_ALPHA,
            zorder=4
        )


# =========================================================
# 7.1 0% null effect 虚线
# =========================================================

if NULL_LINE_MODE == "contour":
    draw_null_effect_contour(
        ax=ax,
        X=X,
        Y=Y,
        z_plot=z_plot,
        color=NULL_CONTOUR_COLOR,
        linewidth=NULL_CONTOUR_LINEWIDTH,
        linestyle=NULL_CONTOUR_LINESTYLE,
        alpha=NULL_CONTOUR_ALPHA
    )
else:
    raise ValueError("当前版本建议使用 NULL_LINE_MODE = 'contour'")


# =========================================================
# 7.2 点阵
# =========================================================

if SIGNIF_POINT_MODE == "original_ci":

    if SHOW_ALL_ORIGINAL_GRID_POINTS:
        ax.scatter(
            df["lag"],
            df["percentile"],
            s=ALL_ORIGINAL_DOT_SIZE,
            color="0.55",
            alpha=ALL_ORIGINAL_DOT_ALPHA,
            zorder=4
        )

    if len(sig_df) > 0:
        ax.scatter(
            sig_df["lag"],
            sig_df["percentile"],
            s=ORIGINAL_SIGNIF_DOT_SIZE,
            color="black",
            zorder=5
        )

elif SIGNIF_POINT_MODE == "interpolated_ci":

    if SHOW_INTERPOLATED_CANDIDATE_DOTS and len(interp_candidate_df) > 0:
        ax.scatter(
            interp_candidate_df["lag"],
            interp_candidate_df["percentile"],
            s=INTERPOLATED_CANDIDATE_DOT_SIZE,
            color="0.55",
            alpha=INTERPOLATED_CANDIDATE_DOT_ALPHA,
            zorder=4
        )

    if len(interp_sig_df) > 0:
        ax.scatter(
            interp_sig_df["lag"],
            interp_sig_df["percentile"],
            s=INTERPOLATED_SIGNIF_DOT_SIZE,
            color="black",
            zorder=5
        )


# =========================================================
# 7.3 坐标轴、标题、边框
# =========================================================

ax.set_xlabel("Lag day", fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("CEHWI percentile", fontsize=AXIS_LABEL_SIZE)

ax.set_xlim(float(np.nanmin(x_plot)), float(np.nanmax(x_plot)))
ax.set_ylim(float(np.nanmin(y_plot)), float(np.nanmax(y_plot)))

ax.set_xticks(np.arange(int(np.floor(x_lag.min())), int(np.ceil(x_lag.max())) + 1, 1))
ax.set_yticks(PERCENTILES)
ax.set_yticklabels([f"p{p}" for p in PERCENTILES])

ax.tick_params(axis="both", labelsize=TICK_SIZE, length=4.5, width=0.9)

ax.set_title(
    "National exposure–lag response surface",
    fontsize=TITLE_SIZE,
    pad=23,
    weight="bold"
)

ax.text(
    0.5,
    1.035,
    "CEHWI | Compound | All activity",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=SUBTITLE_SIZE
)

if SHOW_PANEL_LABEL:
    ax.text(
        -0.12,
        1.105,
        PANEL_LABEL,
        transform=ax.transAxes,
        fontsize=PANEL_LABEL_SIZE,
        weight="bold",
        ha="left",
        va="top"
    )

for spine in ax.spines.values():
    spine.set_linewidth(1.0)


# =========================================================
# 8. Colorbar：连续色带，标注 5 个数字
# =========================================================

if COLORBAR_AS_CONTINUOUS_GRADIENT:
    sm = plt.cm.ScalarMappable(
        cmap=cmap_for_cbar,
        norm=norm_for_cbar
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax,
        extend=COLORBAR_EXTEND
    )
else:
    cbar = fig.colorbar(
        mesh,
        cax=cax,
        extend=COLORBAR_EXTEND
    )

if COLORBAR_TICK_VALUES is not None:
    tick_values = np.asarray(COLORBAR_TICK_VALUES, dtype=float)
else:
    tick_values = np.linspace(COLOR_VMIN, COLOR_VMAX, COLORBAR_N_TICKS)

cbar.set_ticks(tick_values)
cbar.ax.set_yticklabels([
    format_color_label(v) for v in tick_values
])

cbar.ax.set_title(
    "Change in\nphysical activity (%)",
    fontsize=CBAR_LABEL_SIZE,
    pad=8
)

cbar.ax.tick_params(labelsize=TICK_SIZE, length=3.5, width=0.8)
cbar.outline.set_linewidth(0.9)


# =========================================================
# 9. Legend：放在 colorbar 下方，避免重叠
# =========================================================

legend_elements = [
    Line2D(
        [0], [0],
        color=NULL_CONTOUR_COLOR,
        lw=NULL_CONTOUR_LINEWIDTH,
        linestyle=NULL_CONTOUR_LINESTYLE,
        label=NULL_EFFECT_LEGEND_LABEL
    )
]

if SIGNIF_POINT_MODE == "interpolated_ci" and SHOW_INTERPOLATED_CANDIDATE_DOTS:
    legend_elements.append(
        Line2D(
            [0], [0],
            marker="o",
            color="0.55",
            markerfacecolor="0.55",
            alpha=INTERPOLATED_CANDIDATE_DOT_ALPHA,
            linestyle="None",
            markersize=4.0,
            label="Interpolated test grid"
        )
    )

if SIGNIF_POINT_MODE == "original_ci" and SHOW_ALL_ORIGINAL_GRID_POINTS:
    legend_elements.append(
        Line2D(
            [0], [0],
            marker="o",
            color="0.55",
            markerfacecolor="0.55",
            alpha=ALL_ORIGINAL_DOT_ALPHA,
            linestyle="None",
            markersize=4.0,
            label="Original estimated grid point"
        )
    )

if SIGNIF_POINT_MODE == "interpolated_ci":
    signif_label = INTERPOLATED_SIGNIF_LABEL
else:
    signif_label = ORIGINAL_SIGNIF_LABEL

legend_elements.append(
    Line2D(
        [0], [0],
        marker="o",
        color="black",
        linestyle="None",
        markersize=4.2,
        label=signif_label
    )
)

leg_ax.legend(
    handles=legend_elements,
    frameon=False,
    fontsize=LEGEND_FONT_SIZE,
    loc="upper left",
    bbox_to_anchor=(0.0, 1.0),
    borderaxespad=0,
    handlelength=LEGEND_HANDLE_LENGTH,
    handletextpad=LEGEND_HANDLE_TEXT_PAD,
    labelspacing=LEGEND_LABEL_SPACING
)

fig.subplots_adjust(
    left=FIG_LEFT,
    right=FIG_RIGHT,
    bottom=FIG_BOTTOM,
    top=FIG_TOP
)


# =========================================================
# 10. 保存 SVG 和 PNG
# =========================================================

print("Step 6/7: saving figure...")

fig.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight", pad_inches=0.04)
fig.savefig(OUT_SVG, bbox_inches="tight", pad_inches=0.04)
plt.close(fig)


# =========================================================
# 11. 输出信息
# =========================================================

print("Step 7/7: done.")

print("Figure saved:")
print(OUT_PNG)
print(OUT_SVG)

if SAVE_SOURCE_SUMMARY_CSV:
    print("Source data saved:")
    print(source_csv)

if SAVE_LARGE_SURFACE_CSV:
    print("Interpolated response surface saved:")
    print(surface_csv)

if kriging_var_csv is not None:
    print("Kriging variance surface saved:")
    print(kriging_var_csv)

if SIGNIF_POINT_MODE == "interpolated_ci" and SAVE_INTERPOLATED_CI_POINTS_CSV:
    print(f"Interpolated {int(CI_LEVEL_FOR_DOTS * 100)}% CI point table saved:")
    print(interp_points_csv)

if color_break_csv is not None:
    print("Color boundaries saved:")
    print(color_break_csv)

print("Interpolation method:")
print(INTERPOLATION_METHOD)

print("Grid mode:")
print(GRID_MODE)

print("Main interpolation grid:")
print(f"  Lag grid points: {len(x_plot)}")
print(f"  Percentile grid points: {len(y_plot)}")
print(f"  Total surface cells: {len(x_plot) * len(y_plot)}")

print("Figure size:")
print(FIGSIZE)

print("Font scale:")
print(_FONT_SCALE)

print("Main axis box aspect:")
print(MAIN_AX_BOX_ASPECT)

print("Right panel layout:")
print(f"  width ratio = {RIGHT_PANEL_WIDTH_RATIO}")
print(f"  colorbar box = {CBAR_BOX}")
print(f"  legend box = {LEGEND_BOX}")

print("X-axis padding:")
print(f"  left pad: {X_AXIS_LEFT_PAD}")
print(f"  right pad: {X_AXIS_RIGHT_PAD}")
print(f"  x-axis range: {x_plot.min():.2f} to {x_plot.max():.2f}")

print("Y-axis padding:")
print(f"  lower pad: {Y_AXIS_LOWER_PAD}")
print(f"  upper pad: {Y_AXIS_UPPER_PAD}")
print(f"  y-axis range: {y_plot.min():.2f} to {y_plot.max():.2f}")

print("Color mapping:")
print("  mode = natural_breaks")
print(f"  requested color classes = {N_COLOR_CLASSES}")
print(f"  actual color classes = {len(color_boundaries) - 1}")
print(f"  vmin = {COLOR_VMIN}")
print(f"  vmax = {COLOR_VMAX}")
print(f"  color break source = {COLOR_BREAK_SOURCE}")
print(f"  color break max sample = {COLOR_BREAK_MAX_SAMPLE}")
print(f"  colorbar continuous gradient = {COLORBAR_AS_CONTINUOUS_GRADIENT}")
print(f"  colorbar tick values = {COLORBAR_TICK_VALUES}")
print("  color boundaries:")
print(color_boundaries)

print("Significance point mode:")
print(SIGNIF_POINT_MODE)

print("CI level for dots:")
print(CI_LEVEL_FOR_DOTS)

if SIGNIF_POINT_MODE == "interpolated_ci":
    print("Interpolated CI dot grid:")
    print(f"  Lag step: {SIGNIF_GRID_LAG_STEP}")
    print(f"  Percentile step: {SIGNIF_GRID_PERCENTILE_STEP}")
    print(f"  Candidate dots: {len(interp_candidate_df)}")
    print(f"  CI-excludes-null dots: {len(interp_sig_df)}")

print("Null-effect line mode:")
print(NULL_LINE_MODE)
print("Null contour source:")
print("pa_change = 0, equivalent to RR = 1 and log-RR = 0")

print("White contours:")
print(f"  show = {SHOW_WHITE_CONTOURS}")
print(f"  level mode = {WHITE_CONTOUR_LEVEL_MODE}")
print(f"  n levels = {WHITE_CONTOUR_N_LEVELS}")
print(f"  step = {WHITE_CONTOUR_STEP}")
print(f"  exclude zero = {WHITE_CONTOUR_EXCLUDE_ZERO}")
print(f"  alpha = {WHITE_CONTOUR_ALPHA}")

print("Transparency:")
print(f"  heatmap alpha = {HEATMAP_ALPHA}")
print(f"  colorbar alpha = {COLORBAR_ALPHA}")

t1 = time.perf_counter()
print(f"Total runtime: {t1 - t0:.2f} seconds")
print(f"Total runtime: {(t1 - t0) / 60:.2f} minutes")
